<a href="https://colab.research.google.com/github/ElionLAB/OOP_2026_Practice/blob/main/ch_03/src/Lec_3_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Environment Setup — Auto-detect Google Colab / Local
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    pass  # This notebook does not require additional packages.
else:
    print('Local environment: Make sure conda activate OOP is active.')

# Lecture 3: Python Basics II — Data Structures

**Konkuk University OOP (Python Object-Oriented Programming) — Spring 2026**

---

## Learning Objectives

1. Understand when to use built-in structures vs custom classes
2. Analyze immutability vs mutability for design decisions (hashing, state management)
3. Leverage `typing.NamedTuple` and `@dataclass` to reduce boilerplate
4. Master dictionary patterns: views, `defaultdict`, `Counter`, `TypedDict`
5. Evaluate time complexity (O(1) vs O(n)) and memory overhead of different structures
6. Apply structure choices to a real-world case study (k-NN classifier)

---
# Part 1: Slide Code Practice
---

## Practice 1: Empty Objects & `__dict__`
📖 **Slides 4-5**

### 1-1. The Built-in `object()` is Closed (Slide 5)

Instances of `object()` are stateless — they have no `__dict__` and cannot accept dynamic attributes.

In [ ]:
# 1. The built-in object is "closed" to new attributes
o = object()

try:
    o.x = 5  # Raises AttributeError
except AttributeError as e:
    print(f"AttributeError: {e}")

# object() has no __dict__
print(f"hasattr(o, '__dict__'): {hasattr(o, '__dict__')}")

### 1-2. Custom Empty Class Allows Dynamic Attributes (Slide 5)

In [ ]:
# 2. A custom empty class allows dynamic attribute assignment
class Empty:
    pass

e = Empty()
e.x = 5       # Valid: stored in e.__dict__
e.y = "data"

print(f"e.x = {e.x}, e.y = {e.y}")
print(f"e.__dict__ = {e.__dict__}")

### 1-3. Exploring `__dict__`, `hasattr()`, `dir()` (Beyond Slides)

The slide only showed basic attribute assignment. Let's explore the underlying mechanism.

In [ ]:
# __dict__ is a real dictionary — you can manipulate it directly
e.__dict__['z'] = [1, 2, 3]
print(f"e.z = {e.z}")  # Accessible as an attribute!

# hasattr / getattr / delattr
print(f"hasattr(e, 'x'): {hasattr(e, 'x')}")
print(f"hasattr(e, 'w'): {hasattr(e, 'w')}")
print(f"getattr(e, 'w', 'default'): {getattr(e, 'w', 'default')}")

delattr(e, 'z')
print(f"After delattr: {e.__dict__}")

# dir() shows all attributes (including inherited ones)
print(f"Custom attrs in dir: {[a for a in dir(e) if not a.startswith('_')]}")

---
## Practice 2: Tuples as Records
📖 **Slides 6-7**

### 2-1. Tuple as a Record (Slide 7)

Tuples are best understood as **records** (structs) where position indicates meaning.

In [ ]:
# A tuple representing a stock record (Symbol, Current, High, Low)
stock = ("GOOG", 613.30, 625.86, 610.50)

# Unpacking for readability (replaces index access)
symbol, current, high, low = stock
print(f"{symbol}: current={current}, high={high}, low={low}")

# Tuples are immutable
try:
    stock[1] = 615.00
except TypeError as e:
    print(f"TypeError: {e}")

# Valid usage as a dictionary key (because tuples are hashable)
portfolio = {stock: 10}
print(f"portfolio[stock] = {portfolio[stock]}")

### 2-2. Tuple Hashing & Memory Efficiency (Beyond Slides)

Slides mention tuples are hashable and memory-efficient, but don't demonstrate it.

In [ ]:
import sys

# Hashing: tuples are hashable (if contents are hashable)
t = (1, 2, 3)
print(f"hash((1,2,3)) = {hash(t)}")

# A tuple containing a mutable object is NOT hashable
try:
    bad = ([1, 2], 3)
    hash(bad)
except TypeError as e:
    print(f"TypeError: {e}")

# Memory comparison: tuple vs list
data_tuple = tuple(range(100))
data_list = list(range(100))
print(f"tuple size: {sys.getsizeof(data_tuple)} bytes")
print(f"list  size: {sys.getsizeof(data_list)} bytes")
print(f"Tuple is {sys.getsizeof(data_list) - sys.getsizeof(data_tuple)} bytes smaller")

---
## Practice 3: `collections.namedtuple`
📖 **Slides 8-9**

### 3-1. Creating a namedtuple (Slide 9)

`collections.namedtuple` is a factory function that generates a new subclass of `tuple` with named fields.

In [ ]:
from collections import namedtuple

# Create a new class called 'Stock' with specific fields
Stock = namedtuple("Stock", ["symbol", "current", "high", "low"])

# Create an instance
s = Stock("GOOG", 613.30, high=625.86, low=610.50)

# Access by name (Easy to read)
print(f"s.high = {s.high}")

# Access by index (Works like a regular tuple)
print(f"s[2]   = {s[2]}")

# Still immutable
print(f"isinstance(s, tuple): {isinstance(s, tuple)}")

### 3-2. Utility Methods: `_asdict()`, `_replace()`, `_fields` (Beyond Slides)

The slides don't cover these useful built-in methods.

In [ ]:
# _fields — tuple of field names
print(f"Stock._fields = {Stock._fields}")

# _asdict() — convert to an OrderedDict
print(f"s._asdict() = {s._asdict()}")

# _replace() — create a NEW instance with some fields changed
# (Remember: namedtuples are immutable, so this returns a copy)
updated = s._replace(current=620.00)
print(f"Original: {s}")
print(f"Updated:  {updated}")
print(f"Same object? {s is updated}")  # False — it's a new instance

---
## Practice 4: `typing.NamedTuple`
📖 **Slides 10-11**

### 4-1. Modern NamedTuple with Type Hints (Slide 11)

`typing.NamedTuple` provides a class-based syntax with type annotations.

In [ ]:
from typing import NamedTuple

class Stock(NamedTuple):
    symbol: str
    current: float
    high: float
    low: float

    # You can add methods!
    @property
    def spread(self) -> float:
        return self.high - self.low

s = Stock("GOOG", 613.30, 625.86, 610.50)
print(f"Spread: {s.spread:.2f}")  # Output: Spread: 15.36

### 4-2. Default Values & Custom Methods (Beyond Slides)

The slides show `@property` but don't cover default values or regular methods.

In [ ]:
class Stock(NamedTuple):
    symbol: str
    current: float
    high: float = 0.0      # Default value
    low: float = 0.0       # Default value

    @property
    def spread(self) -> float:
        return self.high - self.low

    def is_up(self, open_price: float) -> bool:
        """Check if current price is above the open price."""
        return self.current > open_price

# Using default values
s1 = Stock("MSFT", 305.00)  # high and low default to 0.0
print(f"s1 = {s1}")

s2 = Stock("GOOG", 613.30, 625.86, 610.50)
print(f"s2.is_up(600.00) = {s2.is_up(600.00)}")
print(f"s2.is_up(620.00) = {s2.is_up(620.00)}")

---
## Practice 5: Dataclasses (Mutable)
📖 **Slides 12-13**

### 5-1. Basic Dataclass (Slide 13)

The `@dataclass` decorator auto-generates `__init__`, `__repr__`, `__eq__`, and more.

In [ ]:
from dataclasses import dataclass

@dataclass
class Stock:
    symbol: str
    current: float
    high: float
    low: float

# __init__ is auto-generated
s = Stock("GOOG", 613.30, 625.86, 610.50)

# __repr__ is auto-generated
print(s)

# Dataclasses are mutable
s.current = 614.50
print(f"After update: {s}")

### 5-2. Auto-generated `__eq__` & `__post_init__` (Beyond Slides)

The slides show `__init__` and `__repr__`, but not `__eq__` comparison or `__post_init__`.

In [ ]:
# __eq__ is auto-generated: compares all fields
s1 = Stock("GOOG", 613.30, 625.86, 610.50)
s2 = Stock("GOOG", 613.30, 625.86, 610.50)
s3 = Stock("MSFT", 305.00, 310.00, 300.00)

print(f"s1 == s2: {s1 == s2}")  # True  — same field values
print(f"s1 == s3: {s1 == s3}")  # False — different values
print(f"s1 is s2: {s1 is s2}")  # False — different objects

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Stock:
    symbol: str
    current: float
    high: float
    low: float
    # field() for mutable default values (avoids the mutable default trap)
    history: list = field(default_factory=list)

    def __post_init__(self):
        """Runs after __init__ — useful for validation or computed fields."""
        if self.low > self.high:
            raise ValueError(f"low ({self.low}) cannot be greater than high ({self.high})")

s = Stock("GOOG", 613.30, 625.86, 610.50)
s.history.append(613.30)
s.history.append(614.50)
print(f"{s}")

# Validation in __post_init__
try:
    bad = Stock("BAD", 100.0, high=50.0, low=200.0)
except ValueError as e:
    print(f"ValueError: {e}")

---
## Practice 6: Frozen Dataclasses
📖 **Slides 14-15**

### 6-1. `frozen=True` — Immutable Dataclass (Slide 15)

Setting `frozen=True` prevents attribute modification and enables hashing.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ImmutableStock:
    symbol: str
    current: float

s = ImmutableStock("MSFT", 305.00)

# Attempting to modify raises FrozenInstanceError
try:
    s.current = 310.00
except Exception as e:
    print(f"{type(e).__name__}: {e}")

# Can be used as a dict key because it is hashable
portfolio = {s: 100}
print(f"portfolio[s] = {portfolio[s]}")

### 6-2. Frozen Dataclass vs NamedTuple in Sets (Beyond Slides)

Both are immutable and hashable — let's compare them side by side.

In [ ]:
from typing import NamedTuple
from dataclasses import dataclass

class PointNT(NamedTuple):
    x: int
    y: int

@dataclass(frozen=True)
class PointDC:
    x: int
    y: int

# Both can be used in sets
nt_set = {PointNT(1, 2), PointNT(3, 4), PointNT(1, 2)}  # dedup!
dc_set = {PointDC(1, 2), PointDC(3, 4), PointDC(1, 2)}  # dedup!

print(f"NamedTuple set: {nt_set}")
print(f"Dataclass  set: {dc_set}")

# Key difference: NamedTuple IS a tuple
print(f"\nPointNT(1,2) == (1,2): {PointNT(1,2) == (1,2)}")  # True!
print(f"PointDC(1,2) == (1,2): {PointDC(1,2) == (1,2)}")    # False

---
## Practice 7: Dataclass Ordering
📖 **Slides 16-17**

### 7-1. `order=True` — Automatic Comparisons (Slide 17)

Using `order=True` generates `__lt__`, `__gt__`, etc., allowing natural sorting.

In [ ]:
from dataclasses import dataclass

@dataclass(order=True)
class Stock:
    # Sorts primarily by symbol, then by current price
    symbol: str
    current: float = 0.0  # Default value

s1 = Stock("AAPL", 150.0)
s2 = Stock("GOOG", 2800.0)
s3 = Stock("AAPL", 145.0)

# Automatic comparison logic enabled by order=True
print(f"s1 < s2: {s1 < s2}")   # True ("AAPL" comes before "GOOG")
print(f"s3 < s1: {s3 < s1}")   # True ("AAPL" same, 145.0 < 150.0)

stocks = [s1, s2, s3]
stocks.sort()  # Sorts automatically
print(f"\nSorted: {stocks}")

### 7-2. Sorting with `sort_index` Pattern (Beyond Slides)

What if you want to sort by `current` price only, ignoring `symbol`?

In [ ]:
from dataclasses import dataclass, field

@dataclass(order=True)
class Stock:
    # sort_index is used for comparison, but it's computed, not passed by user
    sort_index: float = field(init=False, repr=False)
    symbol: str = ""
    current: float = 0.0

    def __post_init__(self):
        self.sort_index = self.current  # Sort by price only

stocks = [
    Stock("GOOG", 2800.0),
    Stock("AAPL", 150.0),
    Stock("MSFT", 305.0),
]
stocks.sort()
for s in stocks:
    print(f"{s.symbol}: ${s.current}")

---
## Practice 8: Dictionaries — Views & Iteration
📖 **Slides 18-20**

### 8-1. Dictionary Views are Dynamic (Slide 20)

`.keys()`, `.values()`, `.items()` return **view objects** that update automatically.

In [ ]:
portfolio = {"AAPL": 10, "GOOG": 5}

# .items() returns a dynamic view of key-value pairs
for symbol, shares in portfolio.items():
    print(f"{symbol}: {shares}")

# Dynamic view reflection
keys_view = portfolio.keys()
print(f"\nBefore: {keys_view}")

portfolio["MSFT"] = 100

# The view updates automatically without re-calling .keys()
print(f"After:  {keys_view}")

### 8-2. View Set Operations & Membership Testing (Beyond Slides)

Dictionary views support set-like operations — a powerful feature not shown in the slides.

In [ ]:
portfolio_a = {"AAPL": 10, "GOOG": 5, "MSFT": 20}
portfolio_b = {"GOOG": 8, "TSLA": 15, "MSFT": 20}

# keys() supports set operations
common_stocks = portfolio_a.keys() & portfolio_b.keys()
only_in_a = portfolio_a.keys() - portfolio_b.keys()
print(f"Common stocks: {common_stocks}")
print(f"Only in A:     {only_in_a}")

# items() also supports set operations (checks both key AND value)
same_holdings = portfolio_a.items() & portfolio_b.items()
print(f"Same holdings: {same_holdings}")  # ('MSFT', 20) — same key AND value

# Membership testing is O(1)
print(f"\n'AAPL' in portfolio_a: {'AAPL' in portfolio_a}")

---
## Practice 9: Dictionary Use Cases & TypedDict
📖 **Slides 21-22**

### 9-1. Homogeneous vs Heterogeneous Dictionaries (Slide 22)

In [ ]:
# Use Case 1: Index (Homogeneous) — Symbol maps to Price
stocks = {"GOOG": 1235.20, "MSFT": 110.41}  # dict[str, float]
print(f"GOOG price: {stocks['GOOG']}")

# Use Case 2: Record (Heterogeneous) — Different types
# This works, but NamedTuple or dataclass is often preferred for type safety
record = {"name": "GOOG", "shares": 100, "price": 1235.20}
print(f"Record: {record}")

### 9-2. Dict Comprehension & `setdefault()` (Beyond Slides)

The slides mention `setdefault()` but don't show it in action.

In [ ]:
# Dict comprehension from two lists
symbols = ["AAPL", "GOOG", "MSFT"]
prices = [150.0, 2800.0, 305.0]
stock_dict = {sym: price for sym, price in zip(symbols, prices)}
print(f"stock_dict = {stock_dict}")

# setdefault() — set a value only if the key doesn't exist
groups = {}  # dict[str, list]
words = ["apple", "banana", "avocado", "blueberry", "cherry", "apricot"]

for word in words:
    first_letter = word[0]
    groups.setdefault(first_letter, []).append(word)

print(f"Grouped: {groups}")
# Compare with defaultdict in Practice 10 — same result, cleaner syntax

---
## Practice 10: `defaultdict`
📖 **Slides 23-24**

### 10-1. Grouping with `defaultdict(list)` (Slide 24)

In [ ]:
from collections import defaultdict

# Tell defaultdict to use 'list' for missing keys
grouped_emails = defaultdict(list)

emails = [("work", "boss@corp.com"), ("home", "mom@gmail.com"),
          ("work", "hr@corp.com"), ("home", "dad@gmail.com")]

for tag, email in emails:
    # If 'tag' is new, a new list is created automatically.
    grouped_emails[tag].append(email)

print(dict(grouped_emails))

### 10-2. Word Frequency with `defaultdict(int)` (Beyond Slides)

The slides show `defaultdict(list)` but not `defaultdict(int)` for counting.

In [ ]:
# defaultdict(int) — default value is 0
word_freq = defaultdict(int)

text = "the cat sat on the mat the cat"
for word in text.split():
    word_freq[word] += 1  # No KeyError — starts at 0

print(f"Word frequencies: {dict(word_freq)}")

# Accessing a missing key creates it with default value
print(f"word_freq['dog'] = {word_freq['dog']}")  # 0 (auto-created)
print(f"'dog' in word_freq: {'dog' in word_freq}")  # True! (side effect)

---
## Practice 11: `Counter`
📖 **Slides 25-26**

### 11-1. Counter Basics (Slide 26)

In [ ]:
from collections import Counter

responses = ["yes", "no", "yes", "maybe", "no", "yes"]
counts = Counter(responses)

print(f"counts['yes']     = {counts['yes']}")           # 3
print(f"most_common(1)    = {counts.most_common(1)}")    # [('yes', 3)]

# Missing keys return 0, not KeyError
print(f"counts['unknown'] = {counts['unknown']}")        # 0

# Math with counters
c1 = Counter(a=3, b=1)
c2 = Counter(a=1, b=2)
print(f"c1 + c2 = {c1 + c2}")  # Counter({'a': 4, 'b': 3})

### 11-2. `elements()`, Subtraction & Set Operations (Beyond Slides)

The slides show `most_common()` and addition, but not `elements()` or subtraction.

In [ ]:
c = Counter(a=3, b=2, c=1)

# elements() — iterator over elements repeating each as many times as its count
print(f"elements: {list(c.elements())}")

# Subtraction — removes counts (drops zero and negative)
c1 = Counter(a=4, b=2, c=0)
c2 = Counter(a=2, b=4, d=1)
print(f"c1 - c2 = {c1 - c2}")  # Counter({'a': 2}) — only positive counts

# Intersection (min) and Union (max)
print(f"c1 & c2 = {c1 & c2}")  # Counter({'a': 2, 'b': 2})
print(f"c1 | c2 = {c1 | c2}")  # Counter({'a': 4, 'b': 4, 'd': 1})

# Practical: count characters in a string
char_counts = Counter("mississippi")
print(f"\n'mississippi' counts: {char_counts}")
print(f"Top 3: {char_counts.most_common(3)}")

---
## Practice 12: Lists & Performance
📖 **Slides 27-28**

### 12-1. Homogeneous Lists & Comprehensions (Slide 28)

In [ ]:
# Homogeneous list: correct usage (all floats)
prices = [10.5, 20.0, 15.5]

# List comprehension is Pythonic and fast
taxed_prices = [p * 1.08 for p in prices]
print(f"Taxed prices: {taxed_prices}")

# Heterogeneous list: discouraged
person = ["John", 30, "Engineer"]  # Use a dataclass instead!

### 12-2. `append` vs `insert(0, ...)` Performance (Beyond Slides)

The slides mention O(1) append vs O(n) insert at start, but don't measure it.

In [ ]:
import timeit

# append is O(1) amortized
t_append = timeit.timeit(
    'data.append(99)',
    setup='data = list(range(100_000))',
    number=1000
)

# insert(0, ...) is O(n) — must shift all elements
t_insert = timeit.timeit(
    'data.insert(0, 99)',
    setup='data = list(range(100_000))',
    number=1000
)

print(f"append:    {t_append:.4f} sec")
print(f"insert(0): {t_insert:.4f} sec")
print(f"insert(0) is {t_insert / t_append:.0f}x slower than append")

---
## Practice 13: Sorting
📖 **Slides 29-30**

### 13-1. Sorting with `key` and `attrgetter` (Slide 30)

In [ ]:
from operator import attrgetter

class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price
    def __repr__(self):
        return f"Product({self.name!r}, {self.price})"

products = [Product("Widget", 100), Product("Gadget", 50), Product("Doohickey", 75)]

# Sort by price using a lambda function
products.sort(key=lambda p: p.price)
print(f"By price:  {products}")

# Sort by name using attrgetter (faster and cleaner)
products.sort(key=attrgetter("name"))
print(f"By name:   {products}")

### 13-2. `sorted()` vs `.sort()` & Multi-key Sorting (Beyond Slides)

The slides don't cover the difference between `sorted()` and `.sort()`, or multi-key sorting.

In [ ]:
nums = [3, 1, 4, 1, 5]

# .sort() modifies in-place and returns None
result = nums.sort()
print(f"nums after .sort(): {nums}")
print(f"Return value: {result}")  # None!

# sorted() returns a NEW list, original is unchanged
original = [3, 1, 4, 1, 5]
new_list = sorted(original)
print(f"\noriginal: {original}")  # Unchanged
print(f"new_list: {new_list}")

In [ ]:
# Multi-key sorting with attrgetter
from operator import attrgetter

products = [
    Product("Widget", 100),
    Product("Gadget", 50),
    Product("Widget", 75),   # Same name, different price
    Product("Gadget", 100),  # Same name, different price
]

# Sort by name first, then by price
products.sort(key=attrgetter("name", "price"))
for p in products:
    print(p)

---
## Practice 14: Sets
📖 **Slides 31-32**

### 14-1. Set Deduplication & Algebra (Slide 32)

In [ ]:
# 1. Deduplication
raw_ids = [101, 102, 103, 101, 102]
unique_ids = set(raw_ids)
print(f"Unique IDs: {unique_ids}")

# 2. Set Algebra
admins = {"Alice", "Bob"}
users  = {"Bob", "Charlie", "Dave"}

both = admins & users
regular_users = users - admins
print(f"Both:    {both}")
print(f"Regular: {regular_users}")

# 3. Fast lookup (O(1))
if "Alice" in admins:
    print("Access Granted")

### 14-2. `frozenset`, Unhashable Types, Set of Tuples (Beyond Slides)

The slides mention hashing requirements but don't show `frozenset` or error cases.

In [ ]:
# frozenset — immutable set (can be used as dict key or set element)
fs = frozenset([1, 2, 3])
print(f"frozenset: {fs}")
print(f"hashable:  {hash(fs)}")

# Set of frozensets (nested sets!)
groups = {frozenset([1, 2]), frozenset([3, 4])}
print(f"Set of frozensets: {groups}")

# Set of tuples (tuples are hashable)
coordinates = {(0, 0), (1, 1), (0, 0)}  # dedup!
print(f"Unique coordinates: {coordinates}")

# Lists are NOT hashable — cannot be in a set
try:
    bad_set = {[1, 2], [3, 4]}
except TypeError as e:
    print(f"TypeError: {e}")

---
## Practice 15: Queues — `deque` & `PriorityQueue`
📖 **Slides 33-34**

### 15-1. `deque` and `PriorityQueue` (Slide 34)

In [ ]:
from collections import deque
from queue import PriorityQueue

# 1. Deque: Fast FIFO
dq = deque()
dq.append("Task 1")
dq.append("Task 2")
dq.append("Task 3")
print(f"popleft: {dq.popleft()}")  # Fast O(1). List pop(0) is slow O(n)
print(f"Remaining: {dq}")

# 2. Priority Queue
pq = PriorityQueue()
pq.put((2, "Write Code"))
pq.put((1, "Fix Critical Bug"))
pq.put((3, "Update Docs"))

next_task = pq.get()
print(f"\nHighest priority: {next_task}")  # (1, 'Fix Critical Bug')

### 15-2. `deque` Advanced: `maxlen`, `appendleft`, `rotate` (Beyond Slides)

The slides only show basic `append`/`popleft`. Let's explore more features.

In [ ]:
# maxlen — bounded deque (automatically discards old items)
recent_logs = deque(maxlen=3)
for i in range(5):
    recent_logs.append(f"log_{i}")
    print(f"  Added log_{i}: {list(recent_logs)}")
print(f"Only last 3 kept: {list(recent_logs)}")

# appendleft — add to the left end (O(1))
dq = deque([1, 2, 3])
dq.appendleft(0)
print(f"\nAfter appendleft(0): {dq}")

# rotate — rotate elements
dq = deque([1, 2, 3, 4, 5])
dq.rotate(2)   # Move 2 elements from right to left
print(f"rotate(2):  {dq}")
dq.rotate(-2)  # Move 2 elements from left to right
print(f"rotate(-2): {dq}")

---
## Practice 16: Case Study — K-NN Sample
📖 **Slides 35-36**

### 16-1. Frozen Dataclass for Iris Sample (Slide 36)

A frozen dataclass is chosen because measurement data is immutable once recorded.

In [ ]:
from dataclasses import dataclass

# Chosen Implementation: Frozen Dataclass
@dataclass(frozen=True)
class Sample:
    sepal_len: float
    sepal_width: float
    petal_len: float
    petal_width: float
    species: str

s1 = Sample(5.1, 3.5, 1.4, 0.2, "Iris-setosa")
print(s1)

# Cannot modify (frozen)
try:
    s1.sepal_len = 5.2
except Exception as e:
    print(f"{type(e).__name__}: {e}")

# Hashable: can be used in sets
training_set = {s1}
print(f"Training set size: {len(training_set)}")

### 16-2. Euclidean Distance Between Samples (Beyond Slides)

The slides design the `Sample` class but don't implement the distance function needed for k-NN.

In [ ]:
import math

def euclidean_distance(a: Sample, b: Sample) -> float:
    """Calculate Euclidean distance between two Samples (ignoring species)."""
    return math.sqrt(
        (a.sepal_len - b.sepal_len) ** 2 +
        (a.sepal_width - b.sepal_width) ** 2 +
        (a.petal_len - b.petal_len) ** 2 +
        (a.petal_width - b.petal_width) ** 2
    )

s1 = Sample(5.1, 3.5, 1.4, 0.2, "Iris-setosa")
s2 = Sample(7.0, 3.2, 4.7, 1.4, "Iris-versicolor")
s3 = Sample(5.0, 3.4, 1.5, 0.2, "Iris-setosa")

print(f"dist(s1, s2) = {euclidean_distance(s1, s2):.4f}")  # Far apart
print(f"dist(s1, s3) = {euclidean_distance(s1, s3):.4f}")  # Very close

# Simple 1-NN: classify unknown by nearest training sample
unknown = Sample(5.0, 3.3, 1.4, 0.2, "unknown")
training = [s1, s2, s3]
nearest = min(training, key=lambda s: euclidean_distance(unknown, s))
print(f"\nUnknown classified as: {nearest.species}")

---
# Part 2: Application Problems
---

## Problem 1 (Easy): Refactor Parallel Lists to Dataclass

**Background**: Parallel lists are a common anti-pattern. Refactor them using `@dataclass`.

**Requirements**:
- Create a `Student` dataclass with fields: `name` (str), `age` (int), `major` (str)
- Convert the parallel lists into a `list[Student]`
- Sort the students by age

In [ ]:
from dataclasses import dataclass

# Anti-pattern: parallel lists
names  = ["Alice", "Bob", "Charlie", "Diana"]
ages   = [22, 19, 21, 20]
majors = ["CS", "Math", "CS", "Physics"]


# TODO: Define a Student dataclass


# TODO: Convert parallel lists to a list of Student objects
students = []  # Replace with your code


# TODO: Sort students by age


In [ ]:
# Verification
assert len(students) == 4
assert hasattr(students[0], 'name')
assert hasattr(students[0], 'age')
assert hasattr(students[0], 'major')

# Check sorted by age
for i in range(len(students) - 1):
    assert students[i].age <= students[i + 1].age, "Students should be sorted by age!"

for s in students:
    print(s)

print("\nAll tests passed!")

---
## Problem 2 (Medium): Performance Test — `list` vs `set` Lookup

**Background**: The slides say set lookup is O(1) and list lookup is O(n). Prove it!

**Requirements**:
- Create a `list` and a `set` each containing integers 0 to 999,999
- Use `timeit` to measure the time to check `if 999_999 in collection` (worst case for list)
- Print the results and the speedup ratio

In [ ]:
import timeit

N = 1_000_000
NUMBER = 100  # Number of timeit repetitions

# TODO: Create the list and set
data_list = None  # Replace
data_set = None   # Replace

# TODO: Measure list lookup time (check if N-1 is in data_list)
t_list = 0  # Replace with timeit measurement

# TODO: Measure set lookup time (check if N-1 is in data_set)
t_set = 0   # Replace with timeit measurement


In [ ]:
# Verification
assert data_list is not None, "Create the list!"
assert data_set is not None, "Create the set!"
assert len(data_list) == N
assert len(data_set) == N
assert t_list > 0, "Measure list lookup time!"
assert t_set > 0, "Measure set lookup time!"

print(f"List lookup: {t_list:.6f} sec")
print(f"Set  lookup: {t_set:.6f} sec")
print(f"Set is {t_list / t_set:.0f}x faster than list")

assert t_list > t_set, "Set should be faster than list!"
print("\nAll tests passed!")

---
## Problem 3 (Hard): Implement a `Stack` Class

**Background**: From Slide 39 — implement a generic Stack class using a `list`, then refactor to `collections.deque`.

**Requirements**:
- `push(item)`: Add item to the top
- `pop()`: Remove and return top item (raise `IndexError` if empty)
- `peek()`: Return top item without removing (raise `IndexError` if empty)
- `is_empty()`: Return `True` if stack is empty
- `__len__()`: Return number of items
- `__repr__()`: String representation

In [ ]:
class Stack:
    """A stack (LIFO) implemented with a list."""
    # TODO: Write your code here
    pass


In [ ]:
# Verification
stack = Stack()
assert stack.is_empty() == True
assert len(stack) == 0

stack.push("A")
stack.push("B")
stack.push("C")
print(f"Stack: {stack}")
assert len(stack) == 3

assert stack.peek() == "C"   # Top item
assert len(stack) == 3       # peek doesn't remove

assert stack.pop() == "C"    # Remove top
assert stack.pop() == "B"
assert len(stack) == 1
assert stack.is_empty() == False

assert stack.pop() == "A"
assert stack.is_empty() == True

# Empty stack should raise IndexError
try:
    stack.pop()
except IndexError:
    print("IndexError raised on empty pop (expected)")

try:
    stack.peek()
except IndexError:
    print("IndexError raised on empty peek (expected)")

print("\nAll tests passed!")

### Bonus: Refactor to `deque`

Rewrite the Stack using `collections.deque` instead of `list`. Does the API change?

In [ ]:
from collections import deque

class DequeStack:
    """A stack (LIFO) implemented with deque."""
    # TODO: Write your code here
    pass


In [ ]:
# Verification (same tests, different class)
stack = DequeStack()
stack.push("X")
stack.push("Y")
stack.push("Z")

assert stack.pop() == "Z"
assert stack.peek() == "Y"
assert len(stack) == 2

print(f"DequeStack: {stack}")
print("All tests passed!")

---
## Summary

Topics covered in this lecture:

| Topic | Key Concepts |
|-------|-------------|
| Empty Objects | `object()` is stateless, `class Empty: pass` has `__dict__` |
| Tuples | Records, immutable, hashable, memory-efficient |
| NamedTuple | `collections.namedtuple`, `typing.NamedTuple`, `_asdict()`, `_replace()` |
| Dataclasses | `@dataclass`, `frozen=True`, `order=True`, `field()`, `__post_init__` |
| Dictionaries | Views, `TypedDict`, `defaultdict`, `Counter` |
| Lists | Homogeneous, O(1) append, O(n) insert, list comprehensions |
| Sets | Unique, O(1) lookup, `frozenset`, set algebra |
| Queues | `deque` (O(1) both ends), `PriorityQueue`, `maxlen` |
| Choosing Structures | Intent-driven: immutable → tuple/NamedTuple, mutable → dataclass, lookup → dict/set |

**Next Lecture**: Chapter 4 — Advanced Class Design (Textbook Ch. 3~4)